# CODICE PULITO di visualizzazione_dati_test
1. modifiche file singoli file merge (es. conversioni dummy, round age)
2. merge outer
3. creazione colonna globale per variabile comune ai due dataset
4. studio e applicazione strategia dove ci sono le differenze
5. ricordati ritrasformare dummy e cancellare x e y

- appuntino --> age precedenza PTDEMOG
- da fare prima del merge ---> cancellare "AGE_bl" + dedummyzare
- sistemare "DX" PTDEMOG
- possibile cancellare dopo del merge "VISCODE" == "f"
- controllo both + decisione


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# --- 1. Caricamento dei dataset puliti ---
adnimerge = pd.read_csv("ADNIMERGE_cleaned_02.csv", parse_dates=["EXAMDATE", "update_stamp"])
ptdemog = pd.read_csv("PTDEMOG_cleaned_02.csv", parse_dates=["EXAMDATE", "update_stamp", "PTDOB"])

In [ ]:
adnimerge_originale = pd.read_csv("ADNIMERGE_05Mar2026.csv")
ptdemog_originale = pd.read_csv("PTDEMOG_21Apr2026.csv")

## Riconversione dummys

In [ ]:
cols_x = sorted([c for c in ptdemog.columns if c.endswith('_0')])
coppie = []
for c in cols_x:
    base = c[:-2]
    c_y = base + '_1'
    if c_y in ptdemog.columns:
        coppie.append(base)

print("Coppie trovate:", coppie)

In [ ]:
# --- ADNIMERGE ---
"""
for prefix, categories in [
    ("GENDER", [0, 1]),
    ("MARRY", [0, 1, 2, 3]),
    ("ETHNICITY", [0, 1]),
    ("RACE", [0, 1, 2, 3, 4, 5]),
]:
    cols_dummy = [f"{prefix}_{cat}" for cat in categories]
    ricostruita = pd.from_dummies(adnimerge[cols_dummy], sep="_", default_category="missing")
    adnimerge[prefix] = ricostruita[prefix]
    adnimerge = adnimerge.drop(columns=cols_dummy)"""

In [ ]:
# --- PTDEMOG ---
"""def ricostruisci_categorie(df, lista_categorie):
    cols_dummy = []
    for prefix in lista_categorie:#["GENDER", "MARRY", "ETHNICITY", "RACE"]:
        cols = df.columns[df.columns.str.startswith(prefix)].tolist()
        cols_dummy.extend(cols)
        
        ricostruita = pd.from_dummies(df[cols_dummy], sep="_", default_category="__MISSING__").replace("__MISSING__", np.nan)
        df[prefix] = ricostruita[prefix]
    df = df.drop(columns=cols_dummy)
    return df

adnimerge = ricostruisci_categorie(adnimerge, ['ETHNICITY', 'GENDER', 'MARRY', 'RACE'])
ptdemog = ricostruisci_categorie(ptdemog, ['ETHNICITY', 'GENDER', 'MARRY', 'RACE'])

adnimerge["GENDER", "MARRY", "ETHNICITY", "RACE"]"""

def ricostruisci_categorie(df, lista_categorie):
    df = df.copy()
    tutte_le_dummy = []

    for prefix in lista_categorie:#["GENDER", "MARRY", "ETHNICITY", "RACE"]:
        cols = df.columns[df.columns.str.startswith(prefix)].tolist()
        tutte_le_dummy.extend(cols)   # accumula solo per il drop finale

        ricostruita = pd.from_dummies(df[cols], sep="_", default_category="__MISSING__").replace("__MISSING__", np.nan)
        df[prefix] = ricostruita[prefix]

    df = df.drop(columns=tutte_le_dummy)
    return df

In [ ]:
lista_categorie = ["GENDER", "MARRY", "ETHNICITY", "RACE"]

adnimerge = ricostruisci_categorie(adnimerge, lista_categorie)
ptdemog = ricostruisci_categorie(ptdemog, lista_categorie)

In [ ]:
adnimerge[['GENDER', 'MARRY', 'ETHNICITY', 'RACE']]


## Arrotondamento "AGE"

In [ ]:
adnimerge['AGE'] = adnimerge['AGE'].round(1)
ptdemog['AGE'] = ptdemog['AGE'].round(1)

## Merge

In [ ]:
keys = ["RID", "EXAMDATE"]

# Conto delle combinazioni uniche di chiavi
left_keys = adnimerge[keys].drop_duplicates()
right_keys = ptdemog[keys].drop_duplicates()

key_match = left_keys.merge(
    right_keys,
    on=keys,
    how="outer",
    indicator=True
)

counts = key_match["_merge"].value_counts()
print("Conteggio chiavi uniche per [RID, EXAMDATE]:")
print(counts)

print(f"Match: {counts.get('both', 0)}")
print(f"Solo in adnimerge: {counts.get('left_only', 0)}")
print(f"Solo in ptdemog: {counts.get('right_only', 0)}")

In [ ]:
# --- 2. Merge su RID + EXAMDATE ---
merged = pd.merge(
    adnimerge,
    ptdemog,
    on=keys,
    how="outer",
    indicator=True
)

In [ ]:
# --- 3. Log di controllo post-merge ---
print(merged["_merge"].value_counts())
#merged = merged.drop(columns="_merge") #tenere per le visualizazioni eliminare solo alla fine

In [ ]:
merged = merged[sorted(merged.columns)]
merged

## Creazione colonna unificata

In [ ]:
both_rows = merged[sorted(merged.columns)]
both_rows[both_rows["_merge"] == "both"]

In [ ]:
cols_x = sorted([c for c in both_rows.columns if c.endswith('_x')])
coppie = []
for c in cols_x:
    base = c[:-2]
    c_y = base + '_y'
    if c_y in both_rows.columns:
        coppie.append(base)

print("Coppie trovate:", coppie)

In [ ]:
# --- 2. Confronta ogni coppia, con tolleranza SOLO per le colonne numeriche ---
for base in coppie:
    if base == "update_stamp":
        continue  # non considerare update_stamp per ora

    col_x, col_y = f"{base}_x", f"{base}_y"

    valid_mask = both_rows[col_x].notna() & both_rows[col_y].notna()
    if pd.api.types.is_numeric_dtype(both_rows[col_x]) and pd.api.types.is_numeric_dtype(both_rows[col_y]):
        diff_mask = valid_mask & ((both_rows[col_x] - both_rows[col_y]).abs() > 0.3)   # tolleranza per float
    else:
        diff_mask = valid_mask & (both_rows[col_x] != both_rows[col_y])                # confronto esatto per stringhe/categorie

    n_diff = diff_mask.sum()
    print(f"{base}: {n_diff} righe diverse su {len(both_rows)}")

    if n_diff > 0:
        print(both_rows.loc[diff_mask, ['RID', col_x, col_y]].head(10))
        print()

    # --- Coalesce: priorità a PTDEMOG (_y) solo per AGE, altrimenti priorità ad ADNIMERGE (_x) ---
    if base == "AGE":
        merged[base] = merged[col_y].combine_first(merged[col_x])
    else:
        merged[base] = merged[col_x].combine_first(merged[col_y])

In [ ]:
display_cols = ['RID', 'VISCODE', 'VISCODE_x', 'VISCODE_y', 'VISIT_MONTH', 'EXAMDATE', 'ETHNICITY','RACE', 'AGE', '_merge']
merged[merged["RID"] == 2][display_cols].head(20)

In [ ]:
merged

## Controllo missing data

In [ ]:
missing_report = merged.isna().sum().sort_values(ascending=False)
print(missing_report)

In [ ]:
missing_count = merged.isna().sum()
missing_pct = (missing_count / len(merged) * 100).round(2)

missing_report = pd.DataFrame({
    "n_missing": missing_count,
    "pct_missing": missing_pct
}).sort_values("n_missing", ascending=False)

print(missing_report)

In [ ]:
merged.dtypes

## Ricalcolo "VISCODE" 

In [ ]:
def recompute_visit_month(df, id_col="RID", date_col="EXAMDATE", viscode_col="VISCODE", out_col="VISIT_MONTH"):
    """Ricalcola VISIT_MONTH come mesi trascorsi dalla baseline (bl), per ogni RID."""
    df = df.copy()

    # data di riferimento (bl) per ogni soggetto
    baseline_dates = (
        df.loc[df[viscode_col] == "bl", [id_col, date_col]]
        .drop_duplicates(subset=id_col)
        .set_index(id_col)[date_col]
    )

    df["_baseline_ref"] = df[id_col].map(baseline_dates)
    delta_days = (df[date_col] - df["_baseline_ref"]).dt.days
    df[out_col] = (delta_days / 30.4375).round(0)   # media giorni/mese

    df = df.drop(columns="_baseline_ref")
    return df

In [ ]:
def add_visit_month(df, id_col, date_col):
    """VISIT_MONTH = mesi trascorsi dalla prima visita (baseline) del paziente."""
    df = df.copy().sort_values([id_col, date_col])
    baseline = df.groupby(id_col)[date_col].transform("min")
    df["VISIT_MONTH"] = ((df[date_col] - baseline).dt.days / 30.44).round().astype("Int64")
    return df

In [ ]:
merged = add_visit_month(merged, "RID", "EXAMDATE")

## Ricerca dati mancanti su "ETHNCITY"/"RACE" e soluzione

In [ ]:
display_cols = ['RID', 'ETHNICITY','RACE', 'VISCODE', '_merge']
merged[merged["RACE"] .isna()][display_cols].head(20)

In [ ]:
merged[merged["RACE"].isna()][["RID", "VISCODE", "RACE"]]


In [ ]:
display_cols = ['RID', 'ETHNICITY','RACE', 'VISCODE', '_merge']
merged[merged["RID"] == 10827 ][display_cols].head(20)

In [ ]:
def fill_missing_from_same_subject(df, id_col, target_col):
    """Riempie i NaN di target_col usando il valore noto dello stesso id_col,
    se disponibile in almeno una riga dello stesso soggetto."""
    df = df.copy()
    df[target_col] = df.groupby(id_col)[target_col].transform(
        lambda s: s.ffill().bfill()
    )
    return df

In [ ]:
merged = fill_missing_from_same_subject(merged, "RID", "ETHNICITY")

In [ ]:
merged = fill_missing_from_same_subject(merged, "RID", "RACE")

In [ ]:
merged[merged["RACE"].isna()][["RID", "VISCODE", "ETHNICITY", "RACE"]]

## Considerazione variabili con both

In [ ]:
display_cols = ['RID', 'MARRY','MARRY_x', 'MARRY_y', 'VISCODE', '_merge']
merged[merged["_merge"] == "both" ][display_cols].tail(40)

In [ ]:
display_cols = ['RID', 'MARRY','MARRY_x', 'MARRY_y', 'VISCODE', '_merge', 'update_stamp_x', 'update_stamp_y']
merged[merged["RID"] == 23 ][display_cols]

In [ ]:
def confronta_colonne_merge(df, col_x, col_y, merge_col="_merge"):
     # filtro solo record presenti in entrambi i dataset
    df_both = df[df[merge_col] == "both"].copy()

    # confronto valori diversi (gestione NaN inclusa)
    diff_mask = (
        (df_both[col_x] != df_both[col_y]) &
        ~(df_both[col_x].isna() & df_both[col_y].isna())
    )

    discordanti = df_both.loc[
    diff_mask,
    ["RID", "update_stamp_x", "update_stamp_y", "EXAMDATE", "VISCODE", col_x, col_y, merge_col]
]

    print(f"Record confrontati: {len(df_both)}")
    print(f"Record discordanti: {len(discordanti)}")
    print(f"Percentuale discordanza: {len(discordanti)/len(df_both)*100:.2f}%")

    return discordanti

In [ ]:
discordanti_marry = confronta_colonne_merge(merged,"MARRY_x","MARRY_y")

In [ ]:
display_cols = ["RID", 'MARRY_x', 'MARRY_y',"VISCODE", "EXAMDATE", "update_stamp_x", "update_stamp_y"]
discordanti_marry[display_cols]

In [ ]:
display_cols = ["RID", "MARRY",'MARRY_x', 'MARRY_y', "VISCODE","EXAMDATE", "update_stamp_x", "update_stamp_y"]
merged[merged["MARRY"] == "0"	][display_cols]

In [ ]:
print(ptdemog["GENDER"].dtype)